# Demo: Hybrid OCR evaluation (SROIE & FUNSD)

**Purpose:** Run **PaddleOCR** via the repo's **HybridOCR** class for **SROIE** and **FUNSD** inference. Works on **Google Colab** (Linux; avoids Windows PaddleOCR dependency issues) or **local/SageMaker**. Mode is **auto-detected** (Colab vs local); use **GPU** on Colab to minimize runtime.

**Critical design:** 3-tier detection (cache → classical → **PaddleOCR**) keeps cost and latency low while escalating hard cases; **HybridOCR** unifies Tesseract + PaddleOCR + optional vision fallback. This notebook uses **only OCR deps** (no QNLP/lambeq/jax) so it stays fast and conflict-free.

**Flow:** Setup → instantiate **HybridOCR** → load SROIE/FUNSD from HuggingFace → run OCR → report/download.

## 1. Setup: Colab vs local

On **Google Colab** this cell clones the repo and installs **paddlepaddle** and **paddleocr** (Linux wheels; avoids Windows dependency hell). Then installs the rest of the repo deps so **HybridOCR** and **ocr_pipeline** are available. On **local** (e.g. Linux/SageMaker) ensure you are in the repo root and have already installed paddle + deps.

In [13]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    # Tesseract OCR engine (required by HybridOCR tier 1/2 recognition)
    get_ipython().system("apt-get update -qq && apt-get install -y -qq tesseract-ocr")
    # OCR-only deps: PaddleOCR + pytesseract + anthropic (anthropic needed for ocr_pipeline.recognition import)
    get_ipython().system("pip install -q paddlepaddle paddleocr opencv-python-headless Pillow datasets pytesseract anthropic jedi>=0.16")
    ROOT = Path(".").resolve()
    print("Colab: repo + PaddleOCR + Tesseract ready. HybridOCR will use both when needed.")
else:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "ocr_pipeline").exists() else Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print("Local/SageMaker: ensure paddlepaddle and paddleocr are installed.")

Local/SageMaker: ensure paddlepaddle and paddleocr are installed.


## 2. Instantiate HybridOCR (PaddleOCR in pipeline)

**HybridOCR** uses 3-tier detection (cache → classical → **PaddleOCR**) and Tesseract/PaddleOCR recognition. When the detection router escalates to tier 3 or when recognition falls back to PaddleOCR, the **PaddleOCRDetector** / PaddleOCR engine is used. This cell verifies that PaddleOCR is available (native or ONNX/microservice).

In [14]:
from ocr_pipeline.recognition.hybrid_ocr import HybridOCR
import numpy as np

# 3-tier detection (classical + PaddleOCR fallback); no vision augmentation for speed
ocr_system = HybridOCR(use_detection_router=True, use_vision_augmentation=False)

# Quick check: PaddleOCR detector is loaded by DetectionRouter when available
if hasattr(ocr_system, 'detection_router') and ocr_system.detection_router is not None:
    dr = ocr_system.detection_router
    paddle_ok = getattr(dr, 'paddleocr_available', False)
    print("PaddleOCR available in router:", paddle_ok)
else:
    print("HybridOCR ready (detection router may not have PaddleOCR on first run).")
print("HybridOCR instantiated. Run next cells to run on SROIE/FUNSD.")

✗ No PaddleOCR detection mode available
PaddleOCR available in router: True
HybridOCR instantiated. Run next cells to run on SROIE/FUNSD.


## 2b. OCR proof and score helpers

Defines where proof is written (`data/proof/ocr/`), how to write it (same layout as eval_runner), and simple per-sample scores for SROIE (entity match) and FUNSD (word recall) so you can see evaluation scores in the notebook.

In [15]:
import json
sys.path.insert(0, str(ROOT))
from eval_postprocessing_utils import compute_ocr_metrics, sroie_entity_match_improved, funsd_word_recall_improved

proof_dir = ROOT / "data" / "proof" / "ocr"
proof_dir.mkdir(parents=True, exist_ok=True)

def write_ocr_proof(dataset_name: str, split: str, results: list) -> list:
    """Write per-sample and aggregate proof to data/proof/ocr/<dataset>/<split>/ (same layout as eval_runner). Returns written paths."""
    if not results:
        return []
    ds_lower = dataset_name.lower()
    split_dir = proof_dir / ds_lower / split
    split_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for r in results:
        rows.append({
            "sample_id": r.get("sample_id"),
            "split": split,
            "ground_truth": r.get("ground_truth_entities") or r.get("words_gt") or {},
            "input_text": {},
            "prediction": r.get("text", ""),
            "prediction_error": None,
            "metrics": r.get("metrics", {}),
        })
    samples_path = split_dir / f"{ds_lower}_{split}_samples.json"
    samples_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    sample_count = len(rows)
    split_avg = {"sample_count": sample_count}
    avg_path = split_dir / f"{ds_lower}_{split}_avg.json"
    avg_path.write_text(json.dumps(split_avg, ensure_ascii=False, indent=2), encoding="utf-8")
    dataset_avg_path = proof_dir / ds_lower / f"{ds_lower}_avg.json"
    dataset_avg_path.parent.mkdir(parents=True, exist_ok=True)
    dataset_avg_path.write_text(json.dumps({
        "dataset": dataset_name,
        "sample_count": sample_count,
        "weighted_metrics": {},
        "splits_breakdown": [{"split": split, "sample_count": sample_count}],
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    return [str(samples_path), str(avg_path), str(dataset_avg_path)]

def sroie_entity_score(pred_text: str, gt_entities: dict) -> tuple:
    """Return (matched_count, total_count, details). Uses eval_postprocessing_utils."""
    matched, total, details = sroie_entity_match_improved(pred_text, gt_entities or {}, extract_from_pred=True, normalize=True, soft_cer_threshold=0.35)
    return matched, total, details

def funsd_word_recall(pred_text: str, words_gt: list) -> tuple:
    """Return (recall, n_matched, n_gt). Uses eval_postprocessing_utils (substring + fuzzy)."""
    return funsd_word_recall_improved(pred_text, words_gt or [], normalize=True, use_substring=True, use_fuzzy=True)

print("Proof dir:", proof_dir)
print("Scoring: eval_postprocessing_utils (see data/proof/OCR_LESSONS.md)")

Proof dir: C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\data\proof\ocr
Helpers: write_ocr_proof(dataset_name, split, results), sroie_entity_score(pred, gt_entities), funsd_word_recall(pred, words_gt)


## 4b. OCR proof summary

List proof files under **data/proof/ocr/** and confirm sample counts. Proof is written automatically at the end of the SROIE and FUNSD cells above. Run those cells first, then run this one to verify paths and see how to improve scores.

In [16]:
# Summary of proof written by SROIE and FUNSD cells (run those cells first)
import json
proof_dir = ROOT / "data" / "proof" / "ocr"
print("Proof root:", proof_dir)
print("\nFiles under data/proof/ocr/:")
if not proof_dir.exists():
    print("  (none – run SROIE and FUNSD cells first)")
else:
    for f in sorted(proof_dir.rglob("*")):
        if f.is_file():
            rel = f.relative_to(proof_dir)
            size = f.stat().st_size
            print(f"  {rel} ({size} bytes)")
    for ds in ("sroie", "funsd"):
        samples_path = proof_dir / ds / "train" / f"{ds}_train_samples.json"
        if samples_path.exists():
            try:
                rows = json.loads(samples_path.read_text(encoding="utf-8"))
                n = len(rows)
                print(f"\n{ds.upper()}: {n} samples in {samples_path.name}")
            except Exception as e:
                print(f"\n{ds.upper()}: could not read samples: {e}")
print("\nTo improve scores: check entity_match (SROIE) and word_recall (FUNSD) in the per-sample logs above; adjust OCR pipeline or preprocessing as needed.")

Proof root: C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\data\proof\ocr

Files under data/proof/ocr/:
  funsd\funsd_avg.json (170 bytes)
  funsd\train\funsd_train_avg.json (25 bytes)
  funsd\train\funsd_train_samples.json (16356 bytes)
  sroie\sroie_avg.json (170 bytes)
  sroie\train\sroie_train_avg.json (25 bytes)
  sroie\train\sroie_train_samples.json (2935 bytes)

SROIE: 5 samples in sroie_train_samples.json

FUNSD: 5 samples in funsd_train_samples.json

To improve scores: check entity_match (SROIE) and word_recall (FUNSD) in the per-sample logs above; adjust OCR pipeline or preprocessing as needed.


## 3. Load SROIE and run HybridOCR inference

Load **SROIE** (ICDAR 2019) from HuggingFace (`jsdnrs/ICDAR2019-SROIE`). Default **100 samples** to surface common issues; scoring uses **eval_postprocessing_utils** (entity extraction + normalize + soft CER). See **data/proof/OCR_LESSONS.md** for lessons and postprocessing.

In [17]:
from datasets import load_dataset

SROIE_SPLIT = "train"
MAX_SROIE = 100  # run 100 samples to surface common issues; see data/proof/OCR_LESSONS.md

print("[Step 1] Loading SROIE from HuggingFace ...")
ds_sroie = load_dataset("jsdnrs/ICDAR2019-SROIE", split=SROIE_SPLIT)
sroie_samples = list(ds_sroie.select(range(min(MAX_SROIE, len(ds_sroie)))))
print(f"  Loaded {len(sroie_samples)} samples (split={SROIE_SPLIT}).")

sroie_results = []
verbose = len(sroie_samples) <= 20  # full logs for small runs; progress-only for 100
for i, row in enumerate(sroie_samples):
    sample_id = row.get("key", i)
    if verbose:
        print(f"\n[Step 2] Sample {i+1}/{len(sroie_samples)} (id={sample_id})")
    img = row.get("image")
    if img is None:
        if verbose: print("  Skip: no image")
        continue
    if hasattr(img, "convert"):
        img = img.convert("RGB")
    img_np = np.array(img)
    if verbose: print(f"  Image shape: {img_np.shape}")
    out = ocr_system.process_document(img_np)
    text = out.get("text", "")
    meta = out.get("metadata", {})
    det = meta.get("detection_method", "N/A")
    gt_entities = row.get("entities") or {}
    sroie_results.append({
        "sample_id": sample_id,
        "text": text,
        "ground_truth_entities": gt_entities,
        "metadata": meta,
    })
    matched, total, details = sroie_entity_score(text, gt_entities)
    score_str = f"{matched}/{total}" if total else "N/A"
    if verbose:
        print(f"  [Step 3] Extracted {len(text)} chars; detection: {det}")
        print(f"  [Step 4] Entity score: {score_str}  ({', '.join(details) if details else 'no GT entities'})")
    else:
        if (i + 1) % 10 == 0 or i == 0: print(f"  [Progress] Sample {i+1}/{len(sroie_samples)} (id={sample_id}) chars={len(text)} entity_score={score_str}")
    if total:
        sroie_results[-1]["metrics"] = {"entity_match": matched / total if total else 0, "entity_matched": matched, "entity_total": total}

print("\n" + "=" * 60)
print("SROIE per-sample summary:")
print("  sample_id         | chars | detection  | entity_score")
for r in sroie_results:
    m = r.get("metrics", {})
    es = f'{m.get("entity_matched", 0)}/{m.get("entity_total", 0)}' if m else "N/A"
    print(f"  {str(r.get('sample_id', '')):17} | {len(r.get('text', '')):5} | {r.get('metadata', {}).get('detection_method', 'N/A'):10} | {es}")
if sroie_results and any(r.get("metrics") for r in sroie_results):
    avg_entity = sum(r.get("metrics", {}).get("entity_match", 0) for r in sroie_results) / len(sroie_results)
    print(f"  Average entity match (0-1): {avg_entity:.3f}")
print("=" * 60)

print("\n[Step 5] Writing proof to data/proof/ocr/sroie/ ...")
paths = write_ocr_proof("sroie", SROIE_SPLIT, sroie_results)
for p in paths:
    print(f"  Wrote: {p}")
print(f"Done. {len(sroie_results)} SROIE samples processed and proof saved.")

[Step 1] Loading SROIE from HuggingFace ...
  Loaded 5 samples (split=train).

[Step 2] Sample 1/5 (id=X00016469612)
  Image shape: (1013, 463, 3)
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ...   0   0   0]
 [255 255 255 ...   0   0   0]
 [255 255 255 ...   0   0   0]]
  [Step 3] Extracted 107 chars; detection: paddleocr
  [Step 4] Entity score: 0/4  (company: miss, address: miss, date: miss, total: miss)

[Step 2] Sample 2/5 (id=X00016469619)
  Image shape: (1004, 439, 3)
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
  [Step 3] Extracted 74 chars; detection: paddleocr
  [Step 4] Entity score: 0/4  (company: miss, address: miss, date: miss, total: miss)

[Step 2] Sample 3/5 (id=X00016469620)
  Image shape: (949, 459, 3)
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 2

## 4. Load FUNSD and run HybridOCR inference

Load **FUNSD** (Form Understanding) from HuggingFace (`nielsr/funsd`). Default **100 samples**; scoring uses **eval_postprocessing_utils** (substring + normalize + fuzzy word match). See **data/proof/OCR_LESSONS.md** for lessons and postprocessing.

In [12]:
FUNSD_SPLIT = "train"
MAX_FUNSD = 100  # run 100 samples to surface common issues; see data/proof/OCR_LESSONS.md

print("[Step 1] Loading FUNSD from HuggingFace ...")
ds_funsd = load_dataset("nielsr/funsd", split=FUNSD_SPLIT)
funsd_samples = list(ds_funsd.select(range(min(MAX_FUNSD, len(ds_funsd)))))
print(f"  Loaded {len(funsd_samples)} samples (split={FUNSD_SPLIT}).")

verbose = len(funsd_samples) <= 20  # full logs for small runs; progress-only for 100
funsd_results = []
for i, row in enumerate(funsd_samples):
    sample_id = row.get("id", i)
    if verbose: print(f"\n[Step 2] Sample {i+1}/{len(funsd_samples)} (id={sample_id})")
    img = row.get("image")
    if img is None:
        if verbose: print("  Skip: no image")
        continue
    if hasattr(img, "convert"):
        img = img.convert("RGB")
    img_np = np.array(img)
    if verbose: print(f"  Image shape: {img_np.shape}")
    out = ocr_system.process_document(img_np)
    text = out.get("text", "")
    meta = out.get("metadata", {})
    det = meta.get("detection_method", "N/A")
    words_gt = row.get("words", []) or []
    funsd_results.append({
        "sample_id": sample_id,
        "text": text,
        "words_gt": words_gt,
        "metadata": meta,
    })
    recall, n_in, n_gt = funsd_word_recall(text, words_gt)
    if verbose:
        print(f"  [Step 3] Extracted {len(text)} chars; detection: {det}")
        print(f"  [Step 4] Word recall: {recall:.3f} ({n_in}/{n_gt} GT words in prediction)")
    else:
        if (i + 1) % 10 == 0 or i == 0: print(f"  [Progress] Sample {i+1}/{len(funsd_samples)} (id={sample_id}) chars={len(text)} word_recall={recall:.3f} ({n_in}/{n_gt})")
    funsd_results[-1]["metrics"] = {"word_recall": recall, "words_matched": n_in, "words_gt": n_gt}

print("\n" + "=" * 60)
print("FUNSD per-sample summary:")
print("  sample_id | chars | detection  | word_recall (matched/gt)")
for r in funsd_results:
    m = r.get("metrics", {})
    wr = m.get("word_recall", 0)
    n_in, n_gt = m.get("words_matched", 0), m.get("words_gt", 0)
    print(f"  {str(r.get('sample_id', '')):9} | {len(r.get('text', '')):5} | {r.get('metadata', {}).get('detection_method', 'N/A'):10} | {wr:.3f} ({n_in}/{n_gt})")
if funsd_results:
    avg_recall = sum(r.get("metrics", {}).get("word_recall", 0) for r in funsd_results) / len(funsd_results)
    print(f"  Average word recall (0-1): {avg_recall:.3f}")
print("=" * 60)

print("\n[Step 5] Writing proof to data/proof/ocr/funsd/ ...")
paths = write_ocr_proof("funsd", FUNSD_SPLIT, funsd_results)
for p in paths:
    print(f"  Wrote: {p}")
print(f"Done. {len(funsd_results)} FUNSD samples processed and proof saved.")

[Step 1] Loading FUNSD from HuggingFace ...
  Loaded 5 samples (split=train).

[Step 2] Sample 1/5 (id=0)
  Image shape: (1000, 762, 3)
 [  0   0   0 ... 255 255 255]
 [  0   0   0 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
  [Step 3] Extracted 48 chars; detection: paddleocr
  [Step 4] Word recall: 0.045 (5/111 GT words in prediction)

[Step 2] Sample 2/5 (id=1)
  Image shape: (1000, 762, 3)
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
  [Step 3] Extracted 225 chars; detection: paddleocr
  [Step 4] Word recall: 0.123 (19/154 GT words in prediction)

[Step 2] Sample 3/5 (id=2)
  Image shape: (1000, 762, 3)
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
  [Step 3] Extracted 119 chars; detec

## 5. Run eval_runner.py (monitoring metrics)

Run the unified eval runner with `--category ocr` to evaluate SROIE/FUNSD via `run_model` (HybridOCR), record **layout cache hit rate** and **completeness heuristics** per sample, and write:

- `data/proof/monitoring_metrics/layout_fingerprint_cache/<dataset>_<split>_samples.json`
- `data/proof/monitoring_metrics/completeness_heuristics/<dataset>_<split>_samples.json`
- `data/proof/monitoring_metrics.json` (aggregate)

Run this cell **before** the download cell so the monitoring_metrics folder is populated (and included in the zip download).

In [9]:
# Run eval_runner for OCR: populates monitoring_metrics per-sample + monitoring_metrics.json
import subprocess
import sys
os.chdir(ROOT)
result = subprocess.run([
    sys.executable, "eval_runner.py",
    "--category", "ocr",
    "--max_split", "5",
], capture_output=True, text=True, timeout=600)
if result.returncode != 0:
    print("eval_runner.py finished with exit code", result.returncode)
    if result.stderr:
        print("STDERR:", result.stderr)
    if result.stdout:
        print("STDOUT (last 2000 chars):", result.stdout[-2000:])
else:
    if result.stdout:
        print(result.stdout)
    print("eval_runner.py completed. monitoring_metrics/ and monitoring_metrics.json updated.")

eval_runner.py finished with exit code 1


## 6. Save results and download (Colab)

Write a short **report** and download it plus **data/proof/** artifacts: **monitoring_metrics/** (per-sample JSONs and monitoring_metrics.json) and **ocr/** (SROIE/FUNSD evaluation proofs). Run the **eval_runner** cell above first so both folders are populated.

In [10]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sroie_results = sroie_results if 'sroie_results' in dir() else []
funsd_results = funsd_results if 'funsd_results' in dir() else []

report_lines = [
    "PaddleOCR + HybridOCR (SROIE & FUNSD) – run report",
    "=" * 60,
    f"SROIE samples processed: {len(sroie_results)}",
    f"FUNSD samples processed: {len(funsd_results)}",
    "",
]
for r in (sroie_results or [])[:3]:
    report_lines.append(f"SROIE {r.get('sample_id')}: {len(r.get('text', ''))} chars")
for r in (funsd_results or [])[:3]:
    report_lines.append(f"FUNSD {r.get('sample_id')}: {len(r.get('text', ''))} chars")
report_txt = "\n".join(report_lines)
print(report_txt)
# Only save report to models/ on Colab (for download); locally agents read from notebook output
report_path = MODEL_DIR / "report_paddleocr_sroie_funsd.txt"
if IN_COLAB:
    report_path.write_text(report_txt, encoding="utf-8")
    print(f"Report saved to {report_path}")

try:
    from google.colab import files
    import zipfile
    files.download(str(report_path))
    print("Download started: save report_paddleocr_sroie_funsd.txt to models/.")
    PROOF_DIR = ROOT / "data" / "proof"
    MONITORING_DIR = PROOF_DIR / "monitoring_metrics"
    MONITORING_DIR.mkdir(parents=True, exist_ok=True)
    for sub in ("layout_fingerprint_cache", "completeness_heuristics", "adversarial_testing"):
        subdir = MONITORING_DIR / sub
        subdir.mkdir(parents=True, exist_ok=True)
        placeholder = subdir / "_placeholder.json"
        if not any(subdir.iterdir()):
            placeholder.write_text("[]", encoding="utf-8")
    zip_path = ROOT / "proof_artifacts.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        if (PROOF_DIR / "monitoring_metrics.json").exists():
            zf.write(PROOF_DIR / "monitoring_metrics.json", "monitoring_metrics.json")
        if MONITORING_DIR.exists():
            for f in MONITORING_DIR.rglob("*"):
                if f.is_file():
                    arcname = str(f.relative_to(PROOF_DIR)).replace("\\", "/")
                    zf.write(f, arcname)
        OCR_PROOF = PROOF_DIR / "ocr"
        if OCR_PROOF.exists():
            for f in OCR_PROOF.rglob("*"):
                if f.is_file():
                    arcname = str(f.relative_to(PROOF_DIR)).replace("\\", "/")
                    zf.write(f, arcname)
    files.download(str(zip_path))
    print("Download started: proof_artifacts.zip (extract to data/proof/ for monitoring_metrics/, ocr/, and monitoring_metrics.json).")
except ImportError:
    print("Not on Colab. Report not saved to models/ (see output above).")

PaddleOCR + HybridOCR (SROIE & FUNSD) – run report
SROIE samples processed: 5
FUNSD samples processed: 5

SROIE X00016469612: 107 chars
SROIE X00016469619: 74 chars
SROIE X00016469620: 94 chars
FUNSD 0: 48 chars
FUNSD 1: 225 chars
FUNSD 2: 119 chars
Not on Colab. Report not saved to models/ (see output above).
